# Notebook 14 — Column Operations

**Datasets:** `samples.bakehouse.sales_transactions`, `samples.tpch.orders`, `samples.wanderbricks.bookings`  
**Difficulty:** Easy  
**Topics:** `withColumn`, `withColumnRenamed`, `select`, `drop`, `alias`, `cast`, `lit`, `expr`, `when/otherwise`

These are the bread-and-butter DataFrame column manipulation operations every PySpark developer uses daily.

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T

spark = SparkSession.builder.getOrCreate()

transactions = spark.read.table("samples.bakehouse.sales_transactions")
orders = spark.read.table("samples.tpch.orders")
bookings = spark.read.table("samples.wanderbricks.bookings")

print("sales_transactions schema:")
transactions.printSchema()
print("tpch.orders schema:")
orders.printSchema()
print("wanderbricks.bookings schema:")
bookings.printSchema()

## Learn — Column Operations

| Function | What it does |
|----------|-------------|
| `df.withColumn("name", expr)` | Add or replace a column |
| `df.withColumnRenamed("old", "new")` | Rename a column |
| `df.drop("col")` | Remove a column |
| `F.col("c").cast("double")` | Cast a column to a new type (also: "int", "string", "date") |
| `F.lit(42)` | Create a column with a constant value |
| `F.expr("sql expression")` | Use a SQL expression string as a column |
| `F.when(cond, val).when(cond2, val2).otherwise(default)` | If/elif/else for columns |

**Docs:** [Column API](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/column.html) · [PySpark Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html)

> `withColumn` replaces the column if it already exists, otherwise adds it.
> Chain multiple `withColumn` calls to add several columns at once.

In [ ]:
# Run this example first — then solve the problems below.
# NOTE: this example is not a solution to any problem

df = spark.table("samples.bakehouse.sales_transactions")

# Add a calculated column — tax amount (not a problem question)
df.withColumn("tax_amount", F.round(F.col("totalPrice") * 0.1, 2)) \
  .withColumn("price_category",
      F.when(F.col("totalPrice") < 10, "low")
       .when(F.col("totalPrice") < 50, "medium")
       .otherwise("high")
  ).select("transactionID", "totalPrice", "tax_amount", "price_category").show(5)

## Problem 1 — withColumn and cast

On `sales_transactions`, add two new columns using `.withColumn()`:
1. `card_str` — cast `cardNumber` (bigint) to `StringType` using `.cast(T.StringType())`
2. `revenue_double` — cast `totalPrice` (bigint) to `DoubleType`

**Why:** Casting to string is useful before masking card numbers; casting to double enables floating-point arithmetic.

**Expected output columns:** `transactionID`, `cardNumber`, `card_str`, `totalPrice`, `revenue_double`, `paymentMethod`

In [ ]:
result_1 = None  # replace this

In [ ]:
# ── Tests for Problem 1 ─────────────────
assert result_1 is not None, "result_1 is None"
assert hasattr(result_1, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'transactionid' in cols, "Missing column: transactionID"
assert 'cardnumber' in cols, "Missing column: cardNumber"
assert 'card_str' in cols, "Missing column: card_str"
assert 'totalprice' in cols, "Missing column: totalPrice"
assert 'revenue_double' in cols, "Missing column: revenue_double"
assert 'paymentmethod' in cols, "Missing column: paymentMethod"
assert len(cols) == 6, f"Expected exactly 6 columns, got {len(cols)}: {cols}"
cnt = result_1.count()
assert cnt > 0, f"Got 0 rows"
print(f"Problem 1 passed ✓  ({cnt} rows)")

## Problem 2 — Rename and Drop Columns

On `tpch.orders`, rename the verbose TPC-H column names to friendlier names using `.withColumnRenamed()`, then drop columns that aren't needed for analysis.

Rename:
- `o_orderkey` → `order_id`
- `o_custkey` → `customer_id`
- `o_totalprice` → `total_price`

Then drop: `o_comment`, `o_clerk`

**Expected output columns:** `order_id`, `customer_id`, `total_price`, `o_orderdate`, `o_orderstatus`, `o_orderpriority`

In [ ]:
result_2 = None  # replace this

In [ ]:
# ── Tests for Problem 2 ─────────────────
assert result_2 is not None, "result_2 is None"
assert hasattr(result_2, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'order_id' in cols, "Missing column: order_id"
assert 'customer_id' in cols, "Missing column: customer_id"
assert 'total_price' in cols, "Missing column: total_price"
assert 'o_orderdate' in cols, "Missing column: o_orderdate"
assert 'o_orderstatus' in cols, "Missing column: o_orderstatus"
assert 'o_orderpriority' in cols, "Missing column: o_orderpriority"
assert 'o_comment' not in cols, "o_comment should have been dropped"
assert 'o_clerk' not in cols, "o_clerk should have been dropped"
assert len(cols) == 6, f"Expected exactly 6 columns, got {len(cols)}: {cols}"
cnt = result_2.count()
assert cnt > 0, f"Got 0 rows"
print(f"Problem 2 passed ✓  ({cnt} rows)")

## Problem 3 — Conditional Column with when/otherwise

On `sales_transactions`, add a `price_category` column using `F.when().when().otherwise()`:
- `'Low'` if `totalPrice < 20`
- `'Medium'` if `totalPrice` is between 20 and 100 (inclusive)
- `'High'` if `totalPrice > 100`

Then group by `price_category` to count transactions and compute average price.

**Expected output columns:** `price_category`, `transaction_count`, `avg_price`

In [ ]:
result_3 = None  # replace this

In [ ]:
# ── Tests for Problem 3 ─────────────────
assert result_3 is not None, "result_3 is None"
assert hasattr(result_3, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'price_category' in cols, "Missing column: price_category"
assert 'transaction_count' in cols, "Missing column: transaction_count"
assert 'avg_price' in cols, "Missing column: avg_price"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_3.count()
assert cnt > 0, f"Got 0 rows"
print(f"Problem 3 passed ✓  ({cnt} rows)")

## Problem 4 — Literal and Expression Columns

On `wanderbricks.bookings`, add three new columns:
1. `platform` — a constant string `'Wanderbricks'` using `F.lit()`
2. `stay_nights` — number of nights using `F.expr("datediff(check_out, check_in)")`
3. `per_night_price` — `total_amount / stay_nights` (cost per night)


**Expected output columns:** `booking_id`, `platform`, `check_in`, `check_out`, `stay_nights`, `total_amount`, `per_night_price`

In [ ]:
result_4 = None  # replace this

In [ ]:
# ── Tests for Problem 4 ─────────────────
assert result_4 is not None, "result_4 is None"
assert hasattr(result_4, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'booking_id' in cols, "Missing column: booking_id"
assert 'platform' in cols, "Missing column: platform"
assert 'check_in' in cols, "Missing column: check_in"
assert 'check_out' in cols, "Missing column: check_out"
assert 'stay_nights' in cols, "Missing column: stay_nights"
assert 'total_amount' in cols, "Missing column: total_amount"
assert 'per_night_price' in cols, "Missing column: per_night_price"
assert len(cols) == 7, f"Expected exactly 7 columns, got {len(cols)}: {cols}"
cnt = result_4.count()
assert cnt > 0, f"Got 0 rows"
print(f"Problem 4 passed ✓  ({cnt} rows)")

## Problem 5 — Select with Expressions

On `sales_transactions`, use a **single `.select()` call** to build a clean analytical view with renamed and transformed columns:

| Output column | Source |
|---|---|
| `transaction_id` | alias of `transactionID` |
| `transaction_date` | `F.to_date("dateTime")` |
| `product` | as-is |
| `qty` | alias of `quantity` |
| `unit_price` | alias of `unitPrice` |
| `total_price` | alias of `totalPrice` |
| `payment_type` | alias of `paymentMethod` |

**Expected output columns:** `transaction_id`, `transaction_date`, `product`, `qty`, `unit_price`, `total_price`, `payment_type`

In [ ]:
result_5 = None  # replace this

In [ ]:
# ── Tests for Problem 5 ─────────────────
assert result_5 is not None, "result_5 is None"
assert hasattr(result_5, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'transaction_id' in cols, "Missing column: transaction_id"
assert 'transaction_date' in cols, "Missing column: transaction_date"
assert 'product' in cols, "Missing column: product"
assert 'qty' in cols, "Missing column: qty"
assert 'unit_price' in cols, "Missing column: unit_price"
assert 'total_price' in cols, "Missing column: total_price"
assert 'payment_type' in cols, "Missing column: payment_type"
assert len(cols) == 7, f"Expected exactly 7 columns, got {len(cols)}: {cols}"
cnt = result_5.count()
assert cnt > 0, f"Got 0 rows"
print(f"Problem 5 passed ✓  ({cnt} rows)")